# Experiment: 05 Labeled CASALS Point Cloud By Class Reader

Objective:
- Read one CASALS class-labeled point-cloud product from either `transfer_3dep_labels_to_casals` or `extract_refh_ground`.
- Split points by LAS `classification` and keep one per-class table for inspection or export.
- For `extract_refh_ground` products, compute notebook-only `height_above_ground_m = z - local_ground_z_m` from the final IDW DTM.
- Compute notebook-only local neighborhood features from true 3D point-cloud neighborhoods, including volumetric point density, roughness, local Z span, local Z NMAD, and local slope.
- Add an interactive 3D feature view that colors points by class and lets you map any numeric field onto the X, Y, and Z axes.


In [ ]:
from __future__ import annotations

import json
from collections import defaultdict
from pathlib import Path

import ipywidgets as widgets
import laspy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import rasterio
from IPython.display import display
from pyproj import CRS
from scipy.spatial import cKDTree

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
plt.style.use("seaborn-v0_8-whitegrid")

PRODUCT_KIND = "transfer_3dep_labels"
# "transfer_3dep_labels" or "extract_refh_ground"

POINT_CLOUD_NAME = None
# Example transfer: POINT_CLOUD_NAME = "casals_l1b_20241112T165718_001_02.laz"
# Example ground: POINT_CLOUD_NAME = "casals_l1b_20241112T165718_001_02_tentative_ground_idw_filled_snr5_5m_epsg32618_classified_highsnr.las"

DTM_NAME = None
# Example ground: DTM_NAME = "casals_l1b_20241112T165718_001_02_tentative_ground_idw_filled_snr5_5m_epsg32618.tif"

TARGET_CLASSES = None
# Example: TARGET_CLASSES = [1, 2, 7]

DEFAULT_ANALYSIS_CLASSES = [1, 2]
INCLUDE_LOW_OUTLIERS = False

CHUNK_SIZE = 1_000_000
PREVIEW_ROWS = 5
MAX_BOX_PER_GROUP = 50_000
MAX_SCATTER = 100_000
EXPORT_CLASS_CSV = False

ENABLE_LOCAL_NEIGHBOR_FEATURES = True
LOCAL_FEATURE_RADIUS_M = 5.0
LOCAL_FEATURE_MAX_NEIGHBORS = 24
LOCAL_FEATURE_MIN_NEIGHBORS = 6
LOCAL_FEATURE_QUERY_CHUNK_SIZE = 100_000

ENABLE_INTERACTIVE_3D = True
INTERACTIVE_3D_INCLUDE_NOISE = True
MAX_3D_POINTS_TOTAL = 50_000
MAX_3D_POINTS_PER_CLASS = 20_000
EXCLUDE_INVALID_3D_POINTS = True
DEFAULT_3D_AXES_BY_PRODUCT = {
    "extract_refh_ground": ("height_above_ground_m", "roughness_m", "point_density_pts_m3"),
    "transfer_3dep_labels": ("z", "roughness_m", "point_density_pts_m3"),
}

TRANSFER_OUTPUTS_ROOT = Path("../outputs/transfer_3dep_labels_to_casals")
GROUND_POINT_CLOUD_ROOT = Path("../point_cloud_data/extract_refh_ground")
GROUND_OUTPUTS_ROOT = Path("../outputs/extract_refh_ground")

DEFAULT_COLUMNS_BY_PRODUCT = {
    "transfer_3dep_labels": [
        "x",
        "y",
        "z",
        "classification",
        "longitude",
        "latitude",
        "x_original_m",
        "y_original_m",
        "z_original_m",
        "empirical_dz_m",
        "quality_flag",
        "ground_align_inlier",
        "refh_amp",
        "refh_snr",
        "bg_mean",
        "bg_std",
        "refh_thres",
        "good_snr",
        "transfer_status",
        "nearest3dep_dist_m",
        "nearest3dep_class",
        "dominant3dep_class",
        "class_vote_ratio",
        "n_3dep_neighbors",
        "track_num",
        "sweep_num",
        "delta_time",
        "refh_error",
        "sphericity",
        "linearity",
        "planarity",
        "anisotropy",
        "omnivariance",
    ],
    "extract_refh_ground": [
        "x",
        "y",
        "z",
        "classification",
        "longitude",
        "latitude",
        "refh_snr",
        "refh_amp",
        "refh_thres",
        "ground_resid",
        "pulse_index",
        "track_num",
        "sweep_num",
        "sphericity",
        "linearity",
        "planarity",
        "anisotropy",
        "omnivariance",
    ],
}

SUMMARY_COLUMNS_BY_PRODUCT = {
    "transfer_3dep_labels": [
        "z",
        "z_original_m",
        "refh_amp",
        "refh_snr",
        "nearest3dep_dist_m",
        "class_vote_ratio",
        "point_density_pts_m3",
        "roughness_m",
        "local_z_span_m",
        "local_z_nmad_m",
        "local_slope_deg",
        "sphericity",
        "linearity",
        "planarity",
        "anisotropy",
        "omnivariance",
    ],
    "extract_refh_ground": [
        "z",
        "refh_amp",
        "refh_snr",
        "ground_resid",
        "local_ground_z_m",
        "height_above_ground_m",
        "point_density_pts_m3",
        "roughness_m",
        "local_z_span_m",
        "local_z_nmad_m",
        "local_slope_deg",
        "sphericity",
        "linearity",
        "planarity",
        "anisotropy",
        "omnivariance",
    ],
}

CLASS_NAME_MAP = {
    1: "unclassified_or_elevated",
    2: "ground",
    7: "low_noise",
    9: "water",
    18: "high_noise",
    20: "ignored",
}

CLASS_COLOR_MAP = {
    1: "#1f77b4",
    2: "#2ca02c",
    7: "#d62728",
    9: "#17becf",
    18: "#9467bd",
    20: "#8c564b",
}


## Plan

- Resolve one explicit product mode and discover the matching LAS and sidecars.
- Read only the selected columns, chunk by chunk, and build raw per-class tables.
- For ground-extraction products, sample the final DTM at each point and add `local_ground_z_m`, `height_above_ground_m`, and `dtm_sample_valid`.
- Compute notebook-only local neighborhood features from 3D spherical XYZ neighborhoods so the exported tables and 3D view can use them directly.
- Keep both raw per-class tables and a filtered analysis subset, then summarize, preview, plot, and optionally export.
- Add a sampled Plotly 3D view for the `analysis_df` subset, with per-class colors and switchable X/Y/Z feature axes.


In [ ]:
def available_point_clouds(root: Path) -> list[Path]:
    return sorted(list(root.glob("*.las")) + list(root.glob("*.laz")))


def choose_point_cloud(root: Path, point_cloud_name: str | None = None) -> Path:
    clouds = available_point_clouds(root)
    if not clouds:
        raise FileNotFoundError(f"No LAS/LAZ point cloud found under {root.resolve()}")
    if point_cloud_name is None:
        return max(clouds, key=lambda p: p.stat().st_mtime)
    point_cloud_path = root / point_cloud_name
    if not point_cloud_path.exists():
        raise FileNotFoundError(point_cloud_path.resolve())
    return point_cloud_path


def strip_ground_suffix(stem: str) -> str:
    suffix = "_classified_highsnr"
    return stem[:-len(suffix)] if stem.endswith(suffix) else stem


def read_json_if_exists(path: Path | None) -> dict[str, object] | None:
    if path is None or not path.exists():
        return None
    return json.loads(path.read_text(encoding="utf-8"))


def resolve_product_paths(
    product_kind: str,
    point_cloud_name: str | None = None,
    dtm_name: str | None = None,
) -> dict[str, Path | None]:
    if product_kind == "transfer_3dep_labels":
        point_cloud_path = choose_point_cloud(TRANSFER_OUTPUTS_ROOT, point_cloud_name)
        summary_json_path = point_cloud_path.with_name(point_cloud_path.stem + "_summary.json")
        return {
            "point_cloud_path": point_cloud_path,
            "summary_json_path": summary_json_path,
            "dtm_path": None,
            "metadata_json_path": None,
            "export_root": TRANSFER_OUTPUTS_ROOT / "class_tables",
        }

    if product_kind == "extract_refh_ground":
        point_cloud_path = choose_point_cloud(GROUND_POINT_CLOUD_ROOT, point_cloud_name)
        base_stem = strip_ground_suffix(point_cloud_path.stem)
        dtm_path = (GROUND_OUTPUTS_ROOT / dtm_name) if dtm_name else (GROUND_OUTPUTS_ROOT / f"{base_stem}.tif")
        metadata_json_path = GROUND_OUTPUTS_ROOT / f"{base_stem}_metadata.json"
        if not dtm_path.exists():
            raise FileNotFoundError(f"Matching DTM not found: {dtm_path.resolve()}")
        return {
            "point_cloud_path": point_cloud_path,
            "summary_json_path": metadata_json_path if metadata_json_path.exists() else None,
            "dtm_path": dtm_path,
            "metadata_json_path": metadata_json_path if metadata_json_path.exists() else None,
            "export_root": GROUND_OUTPUTS_ROOT / "class_tables",
        }

    raise ValueError(f"Unsupported PRODUCT_KIND: {product_kind}")


def point_cloud_metadata(point_cloud_path: Path) -> dict[str, object]:
    with laspy.open(point_cloud_path) as reader:
        crs_obj = reader.header.parse_crs()
        raw_dims = [dim.name for dim in reader.header.point_format.dimensions]
        columns = ["x", "y", "z"] + [name for name in raw_dims if name not in {"X", "Y", "Z"}]
        extra_columns = list(reader.header.point_format.extra_dimension_names)
        crs_text = crs_obj.to_string() if crs_obj else None
        return {
            "point_count": int(reader.header.point_count),
            "point_format": int(reader.header.point_format.id),
            "crs": crs_text,
            "crs_obj": CRS.from_user_input(crs_text) if crs_text else None,
            "columns": columns,
            "extra_columns": extra_columns,
        }


def existing_columns(available_columns: list[str], requested_columns: list[str]) -> list[str]:
    available = set(available_columns)
    return [col for col in requested_columns if col in available]


def extract_dimension(points: laspy.ScaleAwarePointRecord, name: str) -> np.ndarray:
    if name in {"x", "y", "z"}:
        return np.asarray(getattr(points, name), dtype=np.float64)
    values = getattr(points, name)
    arr = np.asarray(values)
    if arr.dtype.kind == "b":
        return arr.astype(np.uint8)
    return arr


def read_selected_columns_by_class(
    point_cloud_path: Path,
    selected_columns: list[str],
    target_classes: list[int] | None = None,
    chunk_size: int = 1_000_000,
) -> dict[int, pd.DataFrame]:
    data_by_class: dict[int, list[pd.DataFrame]] = defaultdict(list)
    class_filter = None if target_classes is None else {int(v) for v in target_classes}

    with laspy.open(point_cloud_path) as reader:
        for chunk_id, chunk in enumerate(reader.chunk_iterator(chunk_size), start=1):
            cls = np.asarray(chunk.classification, dtype=np.uint8)
            present_classes = np.unique(cls)
            print(f"chunk {chunk_id}: rows={len(cls):,}, classes={present_classes.tolist()}")

            cached_columns = {name: extract_dimension(chunk, name) for name in selected_columns}
            for class_code in present_classes:
                class_code = int(class_code)
                if class_filter is not None and class_code not in class_filter:
                    continue
                mask = cls == class_code
                block = {name: values[mask] for name, values in cached_columns.items()}
                data_by_class[class_code].append(pd.DataFrame(block))

    class_tables: dict[int, pd.DataFrame] = {}
    for class_code, parts in sorted(data_by_class.items()):
        class_tables[class_code] = pd.concat(parts, ignore_index=True)
    return class_tables


def read_dtm_context(dtm_path: Path) -> dict[str, object]:
    with rasterio.open(dtm_path) as src:
        data = src.read(1).astype(np.float64)
        if src.nodata is not None:
            data[np.isclose(data, src.nodata)] = np.nan
        crs_text = src.crs.to_string() if src.crs else None
        return {
            "path": dtm_path,
            "array": data,
            "transform": src.transform,
            "width": src.width,
            "height": src.height,
            "crs": crs_text,
            "crs_obj": CRS.from_user_input(src.crs.to_wkt()) if src.crs else None,
        }


def assert_matching_crs(point_cloud_crs: CRS | None, dtm_crs: CRS | None) -> None:
    if point_cloud_crs is None:
        raise ValueError("Point-cloud CRS is missing; cannot sample DTM safely.")
    if dtm_crs is None:
        raise ValueError("DTM CRS is missing; cannot sample DTM safely.")
    if not point_cloud_crs.equals(dtm_crs):
        raise ValueError(
            "Point-cloud CRS and DTM CRS do not match. "
            f"LAS CRS: {point_cloud_crs.to_string()} | DTM CRS: {dtm_crs.to_string()}"
        )


def finite_mask(*arrays: np.ndarray) -> np.ndarray:
    if not arrays:
        raise ValueError("finite_mask requires at least one array")
    mask = np.ones(np.asarray(arrays[0]).shape[0], dtype=bool)
    for arr in arrays:
        mask &= np.isfinite(np.asarray(arr))
    return mask


def robust_nmad(values: np.ndarray) -> float:
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return float("nan")
    median = np.median(values)
    return float(1.4826 * np.median(np.abs(values - median)))


def sample_dtm_bilinear(dtm_context: dict[str, object], x: np.ndarray, y: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    arr = np.asarray(dtm_context["array"], dtype=np.float64)
    transform = dtm_context["transform"]
    width = int(dtm_context["width"])
    height = int(dtm_context["height"])

    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    col_f, row_f = (~transform) * (x, y)
    u = np.asarray(col_f, dtype=np.float64) - 0.5
    v = np.asarray(row_f, dtype=np.float64) - 0.5

    col0 = np.floor(u).astype(np.int64)
    row0 = np.floor(v).astype(np.int64)
    dc = u - col0
    dr = v - row0

    in_bounds = (col0 >= 0) & (row0 >= 0) & (col0 + 1 < width) & (row0 + 1 < height)
    samples = np.full(x.shape[0], np.nan, dtype=np.float64)
    valid = np.zeros(x.shape[0], dtype=bool)

    idx = np.flatnonzero(in_bounds)
    if idx.size == 0:
        return samples, valid

    r0 = row0[idx]
    c0 = col0[idx]
    r1 = r0 + 1
    c1 = c0 + 1

    q00 = arr[r0, c0]
    q10 = arr[r0, c1]
    q01 = arr[r1, c0]
    q11 = arr[r1, c1]
    finite_neighbors = np.isfinite(q00) & np.isfinite(q10) & np.isfinite(q01) & np.isfinite(q11)

    if not np.any(finite_neighbors):
        return samples, valid

    idx = idx[finite_neighbors]
    r0 = row0[idx]
    c0 = col0[idx]
    r1 = r0 + 1
    c1 = c0 + 1
    t = dc[idx]
    s = dr[idx]

    q00 = arr[r0, c0]
    q10 = arr[r0, c1]
    q01 = arr[r1, c0]
    q11 = arr[r1, c1]
    samples[idx] = (
        (1.0 - t) * (1.0 - s) * q00
        + t * (1.0 - s) * q10
        + (1.0 - t) * s * q01
        + t * s * q11
    )
    valid[idx] = True
    return samples, valid


def attach_ground_metrics_to_class_tables(
    class_tables: dict[int, pd.DataFrame],
    dtm_context: dict[str, object],
) -> dict[int, pd.DataFrame]:
    out: dict[int, pd.DataFrame] = {}
    for class_code, df in class_tables.items():
        df2 = df.copy()
        local_ground_z_m, dtm_sample_valid = sample_dtm_bilinear(
            dtm_context,
            df2["x"].to_numpy(dtype=np.float64),
            df2["y"].to_numpy(dtype=np.float64),
        )
        df2["local_ground_z_m"] = local_ground_z_m
        df2["dtm_sample_valid"] = dtm_sample_valid.astype(np.uint8)
        df2["height_above_ground_m"] = df2["z"].to_numpy(dtype=np.float64) - local_ground_z_m
        out[class_code] = df2
    return out



def compute_local_neighbor_features(
    df: pd.DataFrame,
    radius_m: float,
    max_neighbors: int,
    min_neighbors: int,
    query_chunk_size: int,
) -> pd.DataFrame:
    feature_names = [
        "local_neighbor_count",
        "point_density_pts_m3",
        "roughness_m",
        "local_z_span_m",
        "local_z_nmad_m",
        "local_slope_deg",
        "sphericity",
        "linearity",
        "planarity",
        "anisotropy",
        "omnivariance",
    ]
    out = {name: np.full(len(df), np.nan, dtype=np.float64) for name in feature_names}
    if df.empty:
        return pd.DataFrame(out, index=df.index)
    if any(col not in df.columns for col in ["x", "y", "z"]):
        return pd.DataFrame(out, index=df.index)

    xyz = np.column_stack(
        [
            pd.to_numeric(df["x"], errors="coerce").to_numpy(dtype=np.float64),
            pd.to_numeric(df["y"], errors="coerce").to_numpy(dtype=np.float64),
            pd.to_numeric(df["z"], errors="coerce").to_numpy(dtype=np.float64),
        ]
    )
    valid_xyz = finite_mask(xyz[:, 0], xyz[:, 1], xyz[:, 2])
    if not np.any(valid_xyz):
        return pd.DataFrame(out, index=df.index)

    xyz_valid = xyz[valid_xyz]
    tree = cKDTree(xyz_valid)
    valid_indices = np.flatnonzero(valid_xyz)
    sphere_volume_m3 = float((4.0 / 3.0) * np.pi * float(radius_m) ** 3)

    max_neighbors = max(int(max_neighbors), int(min_neighbors), 1)
    query_chunk_size = max(1, int(query_chunk_size))

    for start in range(0, xyz_valid.shape[0], query_chunk_size):
        end = min(start + query_chunk_size, xyz_valid.shape[0])
        query_xyz = xyz_valid[start:end]
        print(f"local_feature_chunk: {end:,}/{xyz_valid.shape[0]:,}")
        try:
            neighbor_count = np.asarray(
                tree.query_ball_point(query_xyz, r=float(radius_m), return_length=True, workers=-1),
                dtype=np.int32,
            )
        except TypeError:
            neighbor_count = np.asarray(
                [len(ids) for ids in tree.query_ball_point(query_xyz, r=float(radius_m), workers=-1)],
                dtype=np.int32,
            )

        distances, neighbor_idx = tree.query(
            query_xyz,
            k=int(max_neighbors),
            distance_upper_bound=float(radius_m),
            workers=-1,
        )
        if int(max_neighbors) == 1:
            distances = distances[:, None]
            neighbor_idx = neighbor_idx[:, None]
        valid_neighbor_mask = np.isfinite(distances) & (neighbor_idx >= 0) & (neighbor_idx < xyz_valid.shape[0])

        for local_idx in range(query_xyz.shape[0]):
            target_idx = valid_indices[start + local_idx]
            count = int(neighbor_count[local_idx])
            out["local_neighbor_count"][target_idx] = float(count)
            out["point_density_pts_m3"][target_idx] = count / sphere_volume_m3 if sphere_volume_m3 > 0 else np.nan

            if count < int(min_neighbors):
                continue

            neighbor_ids = neighbor_idx[local_idx, valid_neighbor_mask[local_idx]]
            neighborhood_xyz = xyz_valid[neighbor_ids]
            if neighborhood_xyz.shape[0] < int(min_neighbors):
                continue

            neighborhood_z = neighborhood_xyz[:, 2]
            out["local_z_span_m"][target_idx] = float(np.nanmax(neighborhood_z) - np.nanmin(neighborhood_z))
            out["local_z_nmad_m"][target_idx] = robust_nmad(neighborhood_z)

            centroid = neighborhood_xyz.mean(axis=0)
            centered = neighborhood_xyz - centroid
            try:
                eigvals, eigvecs = np.linalg.eigh((centered.T @ centered) / float(neighborhood_xyz.shape[0]))
            except np.linalg.LinAlgError:
                continue

            eigvals = np.clip(np.asarray(eigvals, dtype=np.float64), 0.0, None)
            l1, l2, l3 = eigvals[::-1]
            if not np.isfinite(l1) or l1 <= 0:
                continue

            out["sphericity"][target_idx] = float(l3 / l1)
            out["linearity"][target_idx] = float((l1 - l2) / l1)
            out["planarity"][target_idx] = float((l2 - l3) / l1)
            out["anisotropy"][target_idx] = float((l1 - l3) / l1)
            out["omnivariance"][target_idx] = float(np.cbrt(max(l1 * l2 * l3, 0.0)))

            normal = eigvecs[:, int(np.argmin(eigvals))]
            normal_norm = float(np.linalg.norm(normal))
            if not np.isfinite(normal_norm) or normal_norm <= 0:
                continue
            normal = normal / normal_norm

            center_xyz = query_xyz[local_idx]
            out["roughness_m"][target_idx] = float(abs((center_xyz - centroid) @ normal))
            out["local_slope_deg"][target_idx] = float(
                np.degrees(np.arctan2(np.linalg.norm(normal[:2]), abs(normal[2])))
            )

    return pd.DataFrame(out, index=df.index)


def attach_local_neighbor_features_to_class_tables(
    class_tables: dict[int, pd.DataFrame],
    radius_m: float,
    max_neighbors: int,
    min_neighbors: int,
    query_chunk_size: int,
) -> dict[int, pd.DataFrame]:
    if not class_tables:
        return {}

    combined = combine_class_tables(class_tables)
    feature_df = compute_local_neighbor_features(
        combined,
        radius_m=radius_m,
        max_neighbors=max_neighbors,
        min_neighbors=min_neighbors,
        query_chunk_size=query_chunk_size,
    )
    combined = combined.copy()
    for col in feature_df.columns:
        combined[col] = feature_df[col].to_numpy()

    out: dict[int, pd.DataFrame] = {}
    start = 0
    for class_code in sorted(class_tables):
        n_rows = len(class_tables[class_code])
        out[class_code] = combined.iloc[start : start + n_rows].reset_index(drop=True)
        start += n_rows
    return out


def resolve_analysis_class_codes(
    class_tables: dict[int, pd.DataFrame],
    default_analysis_classes: list[int],
    include_low_outliers: bool,
) -> list[int]:
    codes = [int(code) for code in default_analysis_classes if int(code) in class_tables]
    if include_low_outliers and 7 in class_tables and 7 not in codes:
        codes.append(7)
    if not codes:
        codes = sorted(class_tables)
    return codes


def combine_class_tables(class_tables: dict[int, pd.DataFrame]) -> pd.DataFrame:
    if not class_tables:
        return pd.DataFrame()
    return pd.concat([class_tables[class_code] for class_code in sorted(class_tables)], ignore_index=True)


def sample_per_group(df: pd.DataFrame, group_col: str, max_n: int) -> pd.DataFrame:
    parts = []
    for _, group in df.groupby(group_col, sort=False, observed=False):
        parts.append(group.sample(n=min(len(group), max_n), random_state=42))
    if not parts:
        return df.iloc[0:0].copy()
    return pd.concat(parts, ignore_index=True)


def summarize_class_tables(class_tables: dict[int, pd.DataFrame], stat_columns: list[str]) -> pd.DataFrame:
    rows = []
    for class_code, df in class_tables.items():
        row = {
            "classification": int(class_code),
            "class_name": CLASS_NAME_MAP.get(int(class_code), f"class_{int(class_code)}"),
            "n_points": int(len(df)),
        }
        if "dtm_sample_valid" in df.columns:
            row["dtm_sample_valid_fraction"] = float(pd.to_numeric(df["dtm_sample_valid"], errors="coerce").fillna(0).mean())
        for col in stat_columns:
            if col not in df.columns:
                continue
            values = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
            row[f"{col}_mean"] = float(values.mean()) if values.notna().any() else np.nan
            row[f"{col}_median"] = float(values.median()) if values.notna().any() else np.nan
            row[f"{col}_p95"] = float(values.quantile(0.95)) if values.notna().any() else np.nan
        rows.append(row)
    if not rows:
        return pd.DataFrame(columns=["classification", "class_name", "n_points"])
    return pd.DataFrame(rows).sort_values(["n_points", "classification"], ascending=[False, True]).reset_index(drop=True)


def summary_metrics_table(product_kind: str, summary_payload: dict[str, object] | None) -> pd.DataFrame | None:
    if summary_payload is None:
        return None
    if product_kind == "transfer_3dep_labels":
        return pd.DataFrame(
            {
                "metric": ["casals_points", "dep3_points", "dz_m_added_to_casals"],
                "value": [
                    summary_payload["counts"]["casals_points"],
                    summary_payload["counts"]["dep3_points"],
                    summary_payload["alignment"]["dz_m_added_to_casals"],
                ],
            }
        )
    return pd.DataFrame(
        {
            "metric": ["n_total_records", "n_high_snr", "n_ground_candidates", "valid_dtm_cells"],
            "value": [
                summary_payload["counts"]["n_total_records"],
                summary_payload["counts"]["n_high_snr"],
                summary_payload["counts"]["n_ground_candidates"],
                summary_payload["final_grid"]["valid_dtm_cells"],
            ],
        }
    )


def export_class_tables(class_tables: dict[int, pd.DataFrame], export_root: Path) -> dict[int, Path]:
    export_root.mkdir(parents=True, exist_ok=True)
    exported: dict[int, Path] = {}
    for class_code, df in class_tables.items():
        out_path = export_root / f"class_{class_code}.csv"
        df.to_csv(out_path, index=False)
        exported[class_code] = out_path
    return exported


def get_numeric_axis_candidates(df: pd.DataFrame) -> list[str]:
    if df.empty:
        return []
    cols = []
    for col in df.columns:
        series = pd.to_numeric(df[col], errors="coerce") if df[col].dtype == object else df[col]
        if pd.api.types.is_numeric_dtype(series) and pd.to_numeric(series, errors="coerce").notna().any():
            cols.append(col)
    return cols


def choose_axis_field(preferred: str, available_columns: list[str], used_columns: set[str]) -> str:
    fallback_order = [preferred, "z", "roughness_m", "point_density_pts_m3", "refh_snr", "refh_amp"] + list(available_columns)
    for col in fallback_order:
        if col in available_columns and col not in used_columns:
            return col
    for col in fallback_order:
        if col in available_columns:
            return col
    raise ValueError("No numeric columns are available for 3D axis selection.")


def resolve_default_3d_axes(product_kind: str, numeric_columns: list[str]) -> tuple[str, str, str]:
    preferred_axes = DEFAULT_3D_AXES_BY_PRODUCT.get(product_kind, ("z", "roughness_m", "point_density_pts_m3"))
    used: set[str] = set()
    axes = []
    for preferred in preferred_axes:
        axis_col = choose_axis_field(preferred, numeric_columns, used)
        axes.append(axis_col)
        used.add(axis_col)
    return tuple(axes)


def get_hover_columns(df: pd.DataFrame, axis_fields: tuple[str, str, str]) -> list[str]:
    curated = [
        "refh_snr",
        "refh_amp",
        "height_above_ground_m",
        "point_density_pts_m3",
        "roughness_m",
        "local_z_span_m",
        "local_z_nmad_m",
        "local_slope_deg",
        "sphericity",
        "linearity",
        "planarity",
        "anisotropy",
        "omnivariance",
        "local_neighbor_count",
        "ground_resid",
        "z",
        "track_num",
        "sweep_num",
    ]
    out = []
    for col in curated:
        if col in df.columns and col not in axis_fields and col not in out:
            out.append(col)
    return out


def class_color_lookup(class_codes: list[int]) -> dict[int, str]:
    palette = px.colors.qualitative.Bold + px.colors.qualitative.Safe + px.colors.qualitative.Set2
    out = dict(CLASS_COLOR_MAP)
    palette_idx = 0
    for class_code in sorted({int(v) for v in class_codes}):
        if class_code in out:
            continue
        out[class_code] = palette[palette_idx % len(palette)]
        palette_idx += 1
    return out


def prepare_interactive_3d_frame(
    df: pd.DataFrame,
    x_axis: str,
    y_axis: str,
    z_axis: str,
    product_kind: str,
    exclude_invalid_3d_points: bool,
) -> pd.DataFrame:
    required = [x_axis, y_axis, z_axis, "classification"]
    for col in required:
        if col not in df.columns:
            return df.iloc[0:0].copy()

    out = df.copy()
    if (
        product_kind == "extract_refh_ground"
        and exclude_invalid_3d_points
        and "dtm_sample_valid" in out.columns
        and "height_above_ground_m" in {x_axis, y_axis, z_axis}
    ):
        out = out.loc[pd.to_numeric(out["dtm_sample_valid"], errors="coerce").fillna(0).astype(bool)].copy()

    for axis_col in [x_axis, y_axis, z_axis]:
        out[axis_col] = pd.to_numeric(out[axis_col], errors="coerce")
    out = out.replace([np.inf, -np.inf], np.nan)
    out = out.dropna(subset=[x_axis, y_axis, z_axis]).copy()
    out["classification"] = pd.to_numeric(out["classification"], errors="coerce").astype(int)
    out["class_name"] = out["classification"].map(lambda x: CLASS_NAME_MAP.get(int(x), f"class_{int(x)}"))
    out["class_label"] = out["classification"].map(lambda x: f"{int(x)} ({CLASS_NAME_MAP.get(int(x), f'class_{int(x)}')})")
    return out


def stratified_sample_for_3d(
    df: pd.DataFrame,
    max_points_per_class: int,
    max_points_total: int,
) -> pd.DataFrame:
    if df.empty:
        return df.copy()

    sampled_parts = []
    for _, group in df.groupby("classification", sort=True, observed=False):
        n = min(len(group), int(max_points_per_class))
        sampled_parts.append(group.sample(n=n, random_state=42) if len(group) > n else group.copy())
    sampled = pd.concat(sampled_parts, ignore_index=True)

    if len(sampled) <= int(max_points_total):
        return sampled

    class_counts = sampled["classification"].value_counts().sort_index()
    ideal = class_counts / class_counts.sum() * int(max_points_total)
    alloc = np.floor(ideal).astype(int)
    alloc[alloc < 1] = 1

    while alloc.sum() > int(max_points_total):
        reducible = alloc[alloc > 1]
        if reducible.empty:
            break
        alloc.loc[reducible.idxmax()] -= 1

    remainder = int(max_points_total) - int(alloc.sum())
    fractions = (ideal - np.floor(ideal)).sort_values(ascending=False)
    for class_code in fractions.index:
        if remainder <= 0:
            break
        if alloc.loc[class_code] < class_counts.loc[class_code]:
            alloc.loc[class_code] += 1
            remainder -= 1

    downsampled_parts = []
    for class_code, group in sampled.groupby("classification", sort=True, observed=False):
        n = min(len(group), int(alloc.loc[class_code]))
        downsampled_parts.append(group.sample(n=n, random_state=42) if len(group) > n else group.copy())
    return pd.concat(downsampled_parts, ignore_index=True)


def make_interactive_3d_figure(
    df: pd.DataFrame,
    x_axis: str,
    y_axis: str,
    z_axis: str,
    hover_columns: list[str],
    class_colors: dict[int, str],
) -> go.Figure:
    fig = go.Figure()
    if df.empty:
        fig.add_annotation(text="No points are available for the selected axes.", x=0.5, y=0.5, showarrow=False)
        fig.update_layout(width=1000, height=650)
        return fig

    marker_size = 3 if len(df) <= 15_000 else 2
    marker_opacity = 0.65 if len(df) <= 15_000 else 0.45

    for class_code, group in df.groupby("classification", sort=True, observed=False):
        class_code = int(class_code)
        hover_cols = [col for col in hover_columns if col in group.columns]
        customdata = group[hover_cols].to_numpy() if hover_cols else None
        hover_lines = [
            f"class={class_code} ({CLASS_NAME_MAP.get(class_code, f'class_{class_code}')})",
            f"{x_axis}=%{{x:.3f}}",
            f"{y_axis}=%{{y:.3f}}",
            f"{z_axis}=%{{z:.3f}}",
        ]
        for idx, col in enumerate(hover_cols):
            hover_lines.append(f"{col}=%{{customdata[{idx}]:.3f}}")
        hovertemplate = "<br>".join(hover_lines) + "<extra></extra>"

        fig.add_trace(
            go.Scatter3d(
                x=group[x_axis],
                y=group[y_axis],
                z=group[z_axis],
                mode="markers",
                name=f"{class_code} ({CLASS_NAME_MAP.get(class_code, f'class_{class_code}')})",
                marker={
                    "size": marker_size,
                    "opacity": marker_opacity,
                    "color": class_colors.get(class_code, "#7f7f7f"),
                },
                customdata=customdata,
                hovertemplate=hovertemplate,
            )
        )

    fig.update_layout(
        width=1000,
        height=700,
        template="plotly_white",
        legend={"title": "classification"},
        margin={"l": 0, "r": 0, "t": 40, "b": 0},
        scene={
            "xaxis": {"title": x_axis, "autorange": True},
            "yaxis": {"title": y_axis, "autorange": True},
            "zaxis": {"title": z_axis, "autorange": True},
            "aspectmode": "cube",
        },
        title=f"Interactive 3D feature view: {x_axis} vs {y_axis} vs {z_axis}",
    )
    return fig


In [ ]:
product_paths = resolve_product_paths(PRODUCT_KIND, POINT_CLOUD_NAME, DTM_NAME)
point_cloud_path = product_paths["point_cloud_path"]
summary_json_path = product_paths["summary_json_path"]
dtm_path = product_paths["dtm_path"]
metadata_json_path = product_paths["metadata_json_path"]

point_metadata = point_cloud_metadata(point_cloud_path)
selected_columns = existing_columns(point_metadata["columns"], DEFAULT_COLUMNS_BY_PRODUCT[PRODUCT_KIND])
summary_payload = read_json_if_exists(summary_json_path)
metrics_table = summary_metrics_table(PRODUCT_KIND, summary_payload)

dtm_context = None
if PRODUCT_KIND == "extract_refh_ground":
    dtm_context = read_dtm_context(dtm_path)
    assert_matching_crs(point_metadata["crs_obj"], dtm_context["crs_obj"])

print("PRODUCT_KIND:", PRODUCT_KIND)
print("point_cloud_path:", point_cloud_path.resolve())
print("summary_json_path:", summary_json_path.resolve() if summary_json_path else "not found")
print("metadata_json_path:", metadata_json_path.resolve() if metadata_json_path else "not found")
print("dtm_path:", dtm_path.resolve() if dtm_path else "not used")
print("point_count:", point_metadata["point_count"])
print("point_format:", point_metadata["point_format"])
print("point_cloud_crs:", point_metadata["crs"])
if dtm_context is not None:
    print("dtm_crs:", dtm_context["crs"])
print("selected_columns:", selected_columns)
print(
    "local_neighbor_features:",
    {
        "enabled": ENABLE_LOCAL_NEIGHBOR_FEATURES,
        "radius_m": LOCAL_FEATURE_RADIUS_M,
        "neighbor_definition": "3d_spherical_xyz",
        "max_neighbors": LOCAL_FEATURE_MAX_NEIGHBORS,
        "min_neighbors": LOCAL_FEATURE_MIN_NEIGHBORS,
        "query_chunk_size": LOCAL_FEATURE_QUERY_CHUNK_SIZE,
    },
)

columns_table = pd.DataFrame(
    {
        "column": point_metadata["columns"],
        "kind": ["extra" if col in set(point_metadata["extra_columns"]) else "standard" for col in point_metadata["columns"]],
    }
)
display(columns_table)

if metrics_table is not None:
    display(metrics_table)


In [ ]:
raw_class_tables = read_selected_columns_by_class(
    point_cloud_path=point_cloud_path,
    selected_columns=selected_columns,
    target_classes=TARGET_CLASSES,
    chunk_size=CHUNK_SIZE,
)

if PRODUCT_KIND == "extract_refh_ground":
    raw_class_tables = attach_ground_metrics_to_class_tables(raw_class_tables, dtm_context)

if ENABLE_LOCAL_NEIGHBOR_FEATURES:
    print(
        "Computing local neighborhood features:",
        {
            "radius_m": LOCAL_FEATURE_RADIUS_M,
            "neighbor_definition": "3d_spherical_xyz",
            "max_neighbors": LOCAL_FEATURE_MAX_NEIGHBORS,
            "min_neighbors": LOCAL_FEATURE_MIN_NEIGHBORS,
            "query_chunk_size": LOCAL_FEATURE_QUERY_CHUNK_SIZE,
        },
    )
    raw_class_tables = attach_local_neighbor_features_to_class_tables(
        raw_class_tables,
        radius_m=LOCAL_FEATURE_RADIUS_M,
        max_neighbors=LOCAL_FEATURE_MAX_NEIGHBORS,
        min_neighbors=LOCAL_FEATURE_MIN_NEIGHBORS,
        query_chunk_size=LOCAL_FEATURE_QUERY_CHUNK_SIZE,
    )
    if TARGET_CLASSES is not None:
        print("local_feature_note: neighbor features were computed only from the loaded TARGET_CLASSES subset.")

analysis_class_codes = resolve_analysis_class_codes(
    raw_class_tables,
    default_analysis_classes=DEFAULT_ANALYSIS_CLASSES,
    include_low_outliers=INCLUDE_LOW_OUTLIERS,
)
analysis_class_tables = {class_code: raw_class_tables[class_code].copy() for class_code in analysis_class_codes}
analysis_df = combine_class_tables(analysis_class_tables)

print("loaded_classes:", sorted(raw_class_tables))
print("analysis_classes:", analysis_class_codes)
print("total_loaded_points:", sum(len(df) for df in raw_class_tables.values()))
print("analysis_points:", len(analysis_df))
if PRODUCT_KIND == "extract_refh_ground" and not analysis_df.empty:
    valid_fraction = pd.to_numeric(analysis_df["dtm_sample_valid"], errors="coerce").fillna(0).mean()
    print("analysis_dtm_sample_valid_fraction:", float(valid_fraction))
if ENABLE_LOCAL_NEIGHBOR_FEATURES and not analysis_df.empty:
    roughness_valid_fraction = pd.to_numeric(analysis_df["roughness_m"], errors="coerce").notna().mean()
    density_valid_fraction = pd.to_numeric(analysis_df["point_density_pts_m3"], errors="coerce").notna().mean()
    sphericity_valid_fraction = pd.to_numeric(analysis_df["sphericity"], errors="coerce").notna().mean() if "sphericity" in analysis_df.columns else float("nan")
    print("analysis_roughness_valid_fraction:", float(roughness_valid_fraction))
    print("analysis_density_valid_fraction:", float(density_valid_fraction))
    print("analysis_sphericity_valid_fraction:", float(sphericity_valid_fraction))


In [ ]:
summary_columns = SUMMARY_COLUMNS_BY_PRODUCT[PRODUCT_KIND]
raw_class_summary = summarize_class_tables(raw_class_tables, summary_columns)
analysis_class_summary = summarize_class_tables(analysis_class_tables, summary_columns)

print("raw_class_summary")
display(raw_class_summary)
print("analysis_class_summary")
display(analysis_class_summary)


In [ ]:
for class_code, df in raw_class_tables.items():
    class_name = CLASS_NAME_MAP.get(int(class_code), f"class_{int(class_code)}")
    print(f"class {class_code} ({class_name}): rows={len(df):,}")
    display(df.head(PREVIEW_ROWS))


In [ ]:
if PRODUCT_KIND == "extract_refh_ground" and not analysis_df.empty:
    plot_df = analysis_df.loc[pd.to_numeric(analysis_df["dtm_sample_valid"], errors="coerce").fillna(0).astype(bool)].copy()
    plot_df["class_label"] = plot_df["classification"].map(lambda x: CLASS_NAME_MAP.get(int(x), f"class_{int(x)}"))

    if plot_df.empty:
        print("No valid DTM samples were found in the analysis subset.")
    else:
        fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))

        for class_code, group in plot_df.groupby("classification", sort=True):
            label = f"{class_code} ({CLASS_NAME_MAP.get(int(class_code), 'class')})"
            axes[0].hist(
                pd.to_numeric(group["height_above_ground_m"], errors="coerce").dropna(),
                bins=80,
                alpha=0.45,
                label=label,
            )
        axes[0].set_title("height_above_ground_m histogram")
        axes[0].set_xlabel("meters")
        axes[0].set_ylabel("count")
        axes[0].legend()

        box_df = plot_df[["class_label", "height_above_ground_m"]].dropna()
        box_df = sample_per_group(box_df, "class_label", MAX_BOX_PER_GROUP)
        box_df.boxplot(column="height_above_ground_m", by="class_label", ax=axes[1], grid=False)
        axes[1].set_title("height_above_ground_m by class")
        axes[1].set_xlabel("class")
        axes[1].set_ylabel("meters")

        scatter_df = plot_df[["ground_resid", "height_above_ground_m", "classification"]].dropna()
        if len(scatter_df) > MAX_SCATTER:
            scatter_df = scatter_df.sample(MAX_SCATTER, random_state=42)
        for class_code, group in scatter_df.groupby("classification", sort=True):
            label = f"{class_code} ({CLASS_NAME_MAP.get(int(class_code), 'class')})"
            axes[2].scatter(
                group["ground_resid"],
                group["height_above_ground_m"],
                s=5,
                alpha=0.20,
                edgecolors="none",
                label=label,
            )
        axes[2].set_title("ground_resid vs height_above_ground_m")
        axes[2].set_xlabel("ground_resid (z - preliminary ground)")
        axes[2].set_ylabel("height_above_ground_m")
        axes[2].legend()

        fig.suptitle("")
        fig.tight_layout()
else:
    print("Height-above-ground plots are only produced for PRODUCT_KIND='extract_refh_ground'.")


In [ ]:
interactive_3d_class_tables = {class_code: df.copy() for class_code, df in analysis_class_tables.items()}
if INTERACTIVE_3D_INCLUDE_NOISE and 7 in raw_class_tables and 7 not in interactive_3d_class_tables:
    interactive_3d_class_tables[7] = raw_class_tables[7].copy()

interactive_3d_source_df = combine_class_tables(interactive_3d_class_tables)
interactive_3d_numeric_columns = get_numeric_axis_candidates(interactive_3d_source_df)
interactive_3d_default_axes = None
interactive_3d_hover_columns = []
interactive_3d_class_colors = class_color_lookup(sorted(interactive_3d_class_tables))

if ENABLE_INTERACTIVE_3D and not interactive_3d_source_df.empty and interactive_3d_numeric_columns:
    interactive_3d_default_axes = resolve_default_3d_axes(PRODUCT_KIND, interactive_3d_numeric_columns)
    interactive_3d_hover_columns = get_hover_columns(interactive_3d_source_df, interactive_3d_default_axes)
    print("interactive_3d_classes:", sorted(interactive_3d_class_tables))
    print("interactive_3d_numeric_columns:", interactive_3d_numeric_columns)
    print("interactive_3d_default_axes:", interactive_3d_default_axes)
else:
    print("Interactive 3D view is disabled or no numeric analysis columns are available.")


In [ ]:
if ENABLE_INTERACTIVE_3D and interactive_3d_default_axes is not None:
    x_default, y_default, z_default = interactive_3d_default_axes
    x_axis_widget = widgets.Dropdown(options=interactive_3d_numeric_columns, value=x_default, description="X axis")
    y_axis_widget = widgets.Dropdown(options=interactive_3d_numeric_columns, value=y_default, description="Y axis")
    z_axis_widget = widgets.Dropdown(options=interactive_3d_numeric_columns, value=z_default, description="Z axis")

    x_min_widget = widgets.Text(value="", placeholder="auto", description="X min")
    x_max_widget = widgets.Text(value="", placeholder="auto", description="X max")
    y_min_widget = widgets.Text(value="", placeholder="auto", description="Y min")
    y_max_widget = widgets.Text(value="", placeholder="auto", description="Y max")
    z_min_widget = widgets.Text(value="", placeholder="auto", description="Z min")
    z_max_widget = widgets.Text(value="", placeholder="auto", description="Z max")

    def parse_axis_limit(text_value: str) -> float | None:
        text = str(text_value).strip()
        if not text:
            return None
        return float(text)

    def apply_axis_range(frame: pd.DataFrame, axis_field: str, axis_config: dict[str, object], min_text: str, max_text: str) -> None:
        min_val = parse_axis_limit(min_text)
        max_val = parse_axis_limit(max_text)
        if min_val is None and max_val is None:
            return

        axis_values = pd.to_numeric(frame[axis_field], errors="coerce").to_numpy(dtype=np.float64)
        finite = np.isfinite(axis_values)
        if not np.any(finite):
            return

        auto_min = float(np.nanmin(axis_values[finite]))
        auto_max = float(np.nanmax(axis_values[finite]))
        lo = auto_min if min_val is None else float(min_val)
        hi = auto_max if max_val is None else float(max_val)
        if not np.isfinite(lo) or not np.isfinite(hi) or lo >= hi:
            return

        axis_config["autorange"] = False
        axis_config["range"] = [lo, hi]

    def render_interactive_3d(
        x_axis: str,
        y_axis: str,
        z_axis: str,
        x_min_text: str,
        x_max_text: str,
        y_min_text: str,
        y_max_text: str,
        z_min_text: str,
        z_max_text: str,
    ) -> None:
        frame = prepare_interactive_3d_frame(
            interactive_3d_source_df,
            x_axis=x_axis,
            y_axis=y_axis,
            z_axis=z_axis,
            product_kind=PRODUCT_KIND,
            exclude_invalid_3d_points=EXCLUDE_INVALID_3D_POINTS,
        )
        frame = stratified_sample_for_3d(
            frame,
            max_points_per_class=MAX_3D_POINTS_PER_CLASS,
            max_points_total=MAX_3D_POINTS_TOTAL,
        )
        hover_columns = get_hover_columns(frame, (x_axis, y_axis, z_axis))
        fig = make_interactive_3d_figure(
            frame,
            x_axis=x_axis,
            y_axis=y_axis,
            z_axis=z_axis,
            hover_columns=hover_columns,
            class_colors=interactive_3d_class_colors,
        )

        if not frame.empty:
            scene = {
                "xaxis": {"title": x_axis, "autorange": True},
                "yaxis": {"title": y_axis, "autorange": True},
                "zaxis": {"title": z_axis, "autorange": True},
                "aspectmode": "cube",
            }
            apply_axis_range(frame, x_axis, scene["xaxis"], x_min_text, x_max_text)
            apply_axis_range(frame, y_axis, scene["yaxis"], y_min_text, y_max_text)
            apply_axis_range(frame, z_axis, scene["zaxis"], z_min_text, z_max_text)
            fig.update_layout(scene=scene)

        fig.show()

    axis_controls = widgets.HBox([x_axis_widget, y_axis_widget, z_axis_widget])
    x_range_controls = widgets.HBox([x_min_widget, x_max_widget])
    y_range_controls = widgets.HBox([y_min_widget, y_max_widget])
    z_range_controls = widgets.HBox([z_min_widget, z_max_widget])
    controls = widgets.VBox([axis_controls, x_range_controls, y_range_controls, z_range_controls])
    output = widgets.interactive_output(
        render_interactive_3d,
        {
            "x_axis": x_axis_widget,
            "y_axis": y_axis_widget,
            "z_axis": z_axis_widget,
            "x_min_text": x_min_widget,
            "x_max_text": x_max_widget,
            "y_min_text": y_min_widget,
            "y_max_text": y_max_widget,
            "z_min_text": z_min_widget,
            "z_max_text": z_max_widget,
        },
    )
    display(controls, output)
else:
    print("Interactive 3D view is unavailable for the current analysis subset.")


Output()

In [ ]:
if EXPORT_CLASS_CSV:
    export_root = product_paths["export_root"] / point_cloud_path.stem
    exported_paths = export_class_tables(raw_class_tables, export_root)
    export_table = pd.DataFrame(
        {
            "classification": list(exported_paths.keys()),
            "csv_path": [str(path.resolve()) for path in exported_paths.values()],
        }
    ).sort_values("classification").reset_index(drop=True)
    display(export_table)
else:
    print("Set EXPORT_CLASS_CSV = True to write one CSV per class.")


## Results

- `raw_class_tables[class_code]` holds the full per-point table for each loaded class.
- `analysis_class_tables` applies the default analysis filter, which keeps classes `1` and `2` and excludes class `7` unless `INCLUDE_LOW_OUTLIERS = True`.
- In `extract_refh_ground` mode, `height_above_ground_m` uses the final IDW DTM, while `ground_resid` remains the residual to the preliminary morphology surface.
- The interactive 3D view uses the filtered analysis subset plus class `7` noise when `INTERACTIVE_3D_INCLUDE_NOISE = True`, then samples per class so the browser stays responsive on large products.


## Next steps

- If class `2` is not centered near zero `height_above_ground_m`, revisit the DTM support mask or ground-candidate tolerances in `extract_refh_ground.py`.
- If the 3D view still feels heavy for a particular product, lower `MAX_3D_POINTS_TOTAL` or `MAX_3D_POINTS_PER_CLASS`.
- If the raw tables are reused elsewhere, export them once and do downstream analysis from the CSV sidecars.
